[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/thermodynamics/rubber_band/rubber_band.ipynb)

# Rubber Band Heat Engine

This example computes a rubber band lifting a weight by heat, the Gough–Joule effect. A rubber band's tension comes from its entropy: stretching it lines up its chains, so a stretched band holds less entropy than a slack one at the same temperature, and it pulls back by its temperature times the entropy it would regain. At a fixed stretch its tension is therefore proportional to its absolute temperature. A band holding up a weight and warmed by a lamp pulls harder, shortens and lifts the weight; cooling, it lengthens and lets the weight down again: a heat engine that turns some of the lamp's heat into the weight's potential energy. Stretched quickly, the band warms; let go, it cools. [Feynman's lectures](https://www.feynmanlectures.caltech.edu/I_44.html) open their chapter on the laws of thermodynamics with this band.

The band is a strip of triangles that moves with the rubber, in the plane joined with a direction for the entropy it holds, and each vertex is one point of that algebra, its place and its entropy, `places + entropies * entropy`. Each triangle's internal energy is one function of its corners' points, and its change as a point moves is one plane, the point's potentials, `temperatures * mv.s - forces`: the force on the point in the plane and its temperature along the entropy, read off together. Stretched at fixed entropy, the rubber warms by its own equation of state, so the warmth of a pull is in the triangles that stretch, with nothing handed from the mechanics to the heat. Conduction is the one process inside the band that makes entropy: the temperature's drive, `-(gradients ^ mv.s)`, through a positive conductivity moves heat from warmer vertices to cooler ones, and on a Delaunay mesh the entropy it makes is never negative.

In the notation of thermoelasticity the free energy per area is $\psi(\boldsymbol\varepsilon, T) = \frac{T}{T_0}\,W(\boldsymbol\varepsilon) + c\,(T - T\ln T)$, so the stress $\partial\psi/\partial\boldsymbol\varepsilon$ is proportional to $T$, the entropy $-\partial\psi/\partial T = c\ln T - W/T_0$ falls with the stretch, and the internal energy $\psi + Ts = cT$ holds no elastic part.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import Algebra, NumpyContext, stack
from numga.sparse import spdiag
from examples.util.animation import show_states
from examples.util.topology import Topology
from examples.thermodynamics.rubber_band import render, scenarios

# The plane, and the entropy the band holds, a direction of its own, degenerate, since nothing measures an amount of
# entropy against a length.
ga = Algebra("x+y+") * Algebra("s0")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                   # Scalar+
Point = ga.gatype.antivector()                                                # Antivector-
Plane = ga.gatype.vector()                                                    # Vector+
Force = ga.gatype.vector()                                                    # Vector+
# The planes reading a place's run across the band and up, and a temperature's drive, a slope wedged with the entropy.
Upright = ga.gatype.from_blades("x y")                                        # Vector+
Drive = ga.gatype.from_blades("xs ys")                                        # Bivector+
# The plane's two directions, across and up, and a unit of entropy.
across = mv.x.dual()                                                          # [] Point-
up = mv.y.dual()                                                              # [] Point-
directions = stack((across, up))                                              # [2] Point-
entropy = mv.s.dual()                                                         # [] Point-

## The band at rest

The band hangs from a support by its top row: rows of nearly equilateral triangles one width across, every other row shifted by half a spacing, so that the mesh prefers no diagonal. Its lowest rows are the weight. Lengths are in units of the band's width, temperatures in units of the room's absolute temperature, and forces in units of the band's shear modulus at room temperature times its width; gravity's acceleration is one. Each triangle's area and its corners' gradients are read once, on the band at rest, the shape its strain is measured from.

In [ ]:
# The band at rest: rows of triangles hanging from the support at the origin, the band's above the weight's, and the
# held top row.
n_across, n_band_rows, n_weight_rows = scenarios.n_across, scenarios.n_band_rows, scenarios.n_weight_rows
coordinates = scenarios.lattice(n_across, n_band_rows + n_weight_rows)       # [vertices, 2]
rest = mv(Upright, coordinates).dual().field()                                # Point-[vertices]
topology = Topology.triangulate(rest)
band_length = n_band_rows * np.sqrt(3) / 2 / (n_across - 1)
in_weight = coordinates[topology.elements[2]][..., 1].max(axis=-1) < -band_length + 1e-9   # [faces]
held = np.flatnonzero(coordinates[:, 1] == 0)                                # [held]
corners = topology.incidence[2, 0]                                            # [corners] Scalar+[faces <- vertices]
corner_mean = corners.mean(axis=0)                                            # Scalar+[faces <- vertices]
# Each triangle's sides, and twice its area: two of its sides joined, at a unit of entropy.
sides = topology.boundary[2, 1] * (topology.d[0] * rest)                      # [corners] Point-[faces]
double_areas = (sides[0] & sides[1]) & entropy                                # Scalar-[faces]
areas = double_areas.norm() / 2                                               # Scalar+[faces]
# A corner's gradient: the plane along the facing side and the entropy, over twice the area, which reads one from the
# side to the corner and nothing on the entropy.
gradients = (entropy & sides) / double_areas                                  # [corners] Plane+[faces]
render.draw_band(topology, in_weight, rest);

## The rubber

The rubber is a plane isotropic material: a work form on pairs of displacement gradients, its dilation's traction and its shear's, of two Lamé moduli, a Poisson ratio of a third making the band narrow as it stretches. Bound to each triangle's test and trial corner gradients, it is a map from the corners' displacements to the forces on them, over the triangle's area. The weight's triangles are the same rubber, denser and stiffer. Both moduli are proportional to the temperature, so at a fixed stretch the band's pull is proportional to its temperature.

In [ ]:
# The rubber's Lamé moduli at the room's temperature; each triangle's stiffening, the weight's stiffer, and its density.
lame_lambda, lame_mu = scenarios.lame_lambda, scenarios.lame_mu
stiffening = mv.scalar(np.where(in_weight, scenarios.weight_stiffening, 1.0)[:, None]).field()   # Scalar+[faces]
densities = mv.scalar(np.where(in_weight, scenarios.weight_density, scenarios.density)[:, None]).field()   # Scalar+[faces]
# The tractions of a plane isotropic material: the dilation's, and the shear's, which removes the rotation from the
# displacement gradient; and its work form on pairs of displacement gradients.
dilation_traction = Plane * (Plane & Point)                                   # [] Plane+ <- (Plane+, Plane+, Point-)
direct_shear = (Plane | Plane) * Point.dual_inverse()
shear_traction = 2 * direct_shear - (Plane | (Plane ^ Point.dual_inverse()))   # [] Plane+ <- (Plane+, Plane+, Point-)
material = (lame_lambda * dilation_traction + lame_mu * shear_traction) & Point   # [] Scalar+ <- (Plane+, Plane+, Point-, Point-)
# The work form bound to each triangle's test and trial corner gradients: a map from the corners' displacements to the
# forces on them, over the triangle's area, at the room's temperature.
bound = gradients.batch()                                                     # [corners, faces] Plane+
work = material(bound[:, None], bound[None, :], Point, Point)
elastic = ((Force & Point).solve(work) * (stiffening * areas).batch()).field()   # [corners, corners] (Force+ <- Point-)[faces]
# Each corner's hat owns a third of its triangle: its cell and its mass.
cells = ~corner_mean * areas                                                  # Scalar+[vertices]
masses = ~corner_mean * (densities * areas)                                   # Scalar+[vertices]

## Heat in the band

The heat lives with the places: each vertex's point holds its entropy, spread across each triangle by its corners' hats. The lamp puts heat in while it is on, falling off with the square of the distance from it, so the side and the height nearest the lamp warm most, and the room takes heat out by Newton's law of cooling. Conduction moves heat from warmer vertices to cooler ones and makes none: the temperature's drive at each hat, its gradient wedged with the entropy, is mapped by the rubber's conductivity to the entropy's flux, and read back by the hats it is a symmetric, positive form on the vertices' temperatures whose rows sum to nothing.

In [ ]:
# The heat the rubber holds per area and degree, its conductivity, the room's temperature and its cooling per area and
# degree; the lamp's heat and reach, and its place beside the band.
heat_capacity, conductivity = scenarios.heat_capacity, scenarios.conductivity
room_temperature, cooling = scenarios.room_temperature, scenarios.cooling
lamp_heat, reach = scenarios.lamp_heat, scenarios.reach
lamp = scenarios.lamp_across * across + scenarios.lamp_up * up                 # [] Point-
capacities = heat_capacity * cells                                            # Scalar+[vertices]
# The rubber's conductivity at a unit rate, the same in every direction: each of the plane's directions swept through
# the entropy, read by a drive and driving the entropy along it.
sweeps = directions & entropy                                                 # [2] Flux-
spreading = (sweeps * (sweeps & Drive)).sum(axis=0)                          # [] Flux- <- Drive+
# The conduction: each corner's hat's drive on the entropy, mapped to its flux and read by the other hats for its
# entropy, over the triangle, gathered onto the vertices.
drives = -(gradients ^ mv.s)                                                  # [corners] Drive+[faces]
local = areas * (mv.s & (gradients[:, None] ^ spreading(drives[None, :])))    # [corners, corners] Scalar+[faces]
conduction = conductivity * (~corners[:, None] * spdiag(local) * corners[None, :]).sum(axis=0).sum(axis=0)   # Scalar+[vertices <- vertices]

## The triangles' energy

Each triangle's free energy is its temperature over the room's times its stretch's elastic energy at the room's temperature, and its heat capacity's; its entropy is its heat's less that elastic energy over the room's temperature. So a triangle warms as it stretches at fixed entropy, and its pull is its elastic forces scaled by its warmth. Both are parts of one plane per vertex, the energy's change as the vertex's point moves.

In [ ]:
def potentials(points: Point) -> Plane:
    """Each vertex's potentials `Plane+[vertices]`, its triangles' internal energy's change as its point moves: minus
    the force on it in the plane, and its temperature along the entropy; for the vertices at these points
    `Point-[vertices]`, each its place and its entropy."""
    # Each point's entropy, read by the plane along it, and its place, the point without it.
    held = mv.s & points                                                      # Scalar+[vertices]
    shifts = corners * ((points - held * entropy) - rest)                     # [corners] Point-[faces]
    # The forces of the stretch at the room's temperature, and its elastic energy, half the forces read on the shifts.
    springs = -elastic(shifts[None, :]).sum(axis=1)                           # [corners] Force+[faces]
    stretch = -(springs & shifts).sum(axis=0) / 2                             # Scalar+[faces]
    # The entropy per area each triangle holds, its corners' spread by their hats, and the warmth it and the stretch
    # give: the heat's entropy is the heat capacity times the logarithm of the temperature over the room's.
    held_densities = corner_mean * (held / cells)                             # Scalar+[faces]
    warmth = ((held_densities + stretch / (areas * room_temperature)) / heat_capacity).exp()   # Scalar+[faces]
    # Each vertex's temperature, its triangles' around it by their areas, its cell's share, and the forces on it.
    temperatures = room_temperature * (~corner_mean * (areas * warmth)) / cells   # Scalar+[vertices]
    forces = (~corners * (warmth * springs)).sum(axis=0)                      # Force+[vertices]
    return temperatures * mv.s - forces

## The step

The motion is the reversible half: the vertices move at fixed entropy by the forces the potentials give, taken at the middle of each move, which keeps the energy without drift. The heat is the irreversible half: what the lamp, the room and the conduction move over a step carries each vertex's point along the entropy, a cell's entropy growing by its heat capacity times the logarithm of its temperature's rise.

In [ ]:
# Gravity's pull on a unit of mass, a form on its steps; a hand's pull on the weight at full strength, as hard as the
# weight's own weight again; the air's drag per mass; the time step; and the reading that frees all but the held top row.
pull = -scenarios.gravity * mv.y                                              # [] Plane+
weighting = mv.scalar(in_weight[:, None] * 1.0).field()                       # Scalar+[faces]
hand = scenarios.hand_pull * (~corner_mean * (weighting * densities * areas)) * pull   # Force+[vertices]
drag, dt = scenarios.drag, scenarios.dt
free = topology.reading(np.setdiff1d(np.arange(topology.n_vertices), held))
freeing = ~free * free                                                        # Scalar+[vertices <- vertices]


def move(points: Point, velocities: Point, forces: Force, pulled: float) -> tuple[Point, Point]:
    """The band's points and velocities `Point-[vertices]` a step on from these, moved at fixed entropy by position
    Verlet under these forces of its stretch `Force+[vertices]`, taken at the middle of the move, gravity and the hand
    pulling the weight as much as `pulled`, from none to full; the held top row stays still."""
    # The kick by the forces against the drag of the mean of the velocities before and after it, and the step by that
    # mean.
    loads = forces + masses * pull + pulled * hand                            # Force+[vertices]
    kicked = freeing * (((1 - drag * dt / 2) * velocities + dt * (loads / masses).dual()) / (1 + drag * dt / 2))
    return points + dt * (velocities + kicked) / 2, kicked

In [ ]:
def exchange(points: Point, temperatures: Scalar, lit: float) -> Scalar:
    """The entropy `Scalar+[vertices]` each vertex gains from the heat the lamp, lit as much as `lit`, from none to
    full, the room and the conduction move into its cell over a step, the vertices at these points `Point-[vertices]`
    and these temperatures `Scalar+[vertices]`."""
    # The lamp's exposure falls off with the square of the distance from it, from the point's place, the point without
    # its entropy.
    offsets = points - (mv.s & points) * entropy - lamp                       # Point-[vertices]
    exposures = 1 / (1 + (offsets.dual_inverse() & offsets) / reach ** 2)     # Scalar+[vertices]
    exchanged = dt * cells * (lit * lamp_heat * exposures - cooling * (temperatures - room_temperature))
    conducted = -dt * (conduction * temperatures)                             # Scalar+[vertices]
    # The heat warms a cell by itself over the cell's heat capacity, and its entropy grows by the heat capacity times
    # the logarithm of the warmer temperature over the cooler.
    return capacities * (1 + (exchanged + conducted) / (capacities * temperatures)).log()

In [ ]:
def step(points: Point, velocities: Point, lit: float, pulled: float) -> tuple[Point, Point]:
    """The band's points and velocities `Point-[vertices]` a step on from these: a move at fixed entropy, and the heat
    the lamp, lit as much as `lit`, the room and the conduction move over it carrying each point along the entropy,
    the hand pulling as much as `pulled`."""
    # The potentials at the middle of the move, one plane per vertex: minus the force in the plane, which moves the
    # band, and the temperature along the entropy, at which the heat moves.
    middle = points + dt / 2 * velocities                                     # Point-[vertices]
    middle_potentials = potentials(middle)                                    # Plane+[vertices]
    temperatures = middle_potentials & entropy                                # Scalar+[vertices]
    forces = temperatures * mv.s - middle_potentials                          # Force+[vertices]
    points, velocities = move(points, velocities, forces, pulled)
    gained = exchange(middle, temperatures, lit)                              # Scalar+[vertices]
    return points + gained * entropy, velocities


def run(points: Point, exposure: np.ndarray, pulling: np.ndarray, n_stride: int
        ) -> Iterator[tuple[Point, Scalar, float]]:
    """The band's points `Point-[vertices]`, each its place and its entropy, their temperatures `Scalar+[vertices]` and
    whether the lamp is on, at the start and after every `n_stride` steps, one step for each of the lamp's exposures
    and the hand's pulls `[steps]`, each from none to full, from the band still at these points."""
    velocities = 0 * points                                                   # Point-[vertices]
    yield points, potentials(points) & entropy, exposure[0]
    for exposures, pulls in zip(exposure.reshape(-1, n_stride), pulling.reshape(-1, n_stride)):
        for lit, pulled in zip(exposures, pulls):
            points, velocities = step(points, velocities, lit, pulled)
        yield points, potentials(points) & entropy, exposures[-1]

## Lifting and lowering the weight

The band is let go at rest under the weight and settles with the lamp off. Then the lamp is switched on for two and a half times the time the band takes to warm, and off for as long. The animation is a time-lapse: a real band warms over minutes while the weight would bounce on it within a second, and here the warming is sped up to a few times the bounce's period, which the air's drag keeps from showing. The lamp heats the band far beyond what real rubber bears, so that the lift is plain. This cell takes about fifteen seconds, most of it stepping the band.

In [ ]:
# The run: the steps the band settles under the weight with the lamp off, run but not shown; the steps the lamp is on
# and then off; the steps between frames.
n_settling, n_lit, n_dark, n_stride = scenarios.n_settling, scenarios.n_lit, scenarios.n_dark, scenarios.n_stride

# The band let go at rest at the room's temperature, holding no entropy beyond the room's, settled; its last state
# starts the run.
*_, (settled, *_) = run(rest, np.zeros(n_settling), np.zeros(n_settling), n_settling)
exposure = np.concatenate([np.ones(n_lit), np.zeros(n_dark)])
states = show_states(run(settled, exposure, np.zeros_like(exposure), n_stride), render.animate, topology,
                     in_weight, lamp, room_temperature, scenarios.lamp_temperature, name="rubber_band",
                     duration_ms=scenarios.frame_duration_ms)

The band is grey at the room's temperature, narrowing below its clamp under the weight's pull, and warms from the side and the height nearest the lamp, which glows while it is on, orange and then red. Warmed, it pulls harder and lifts the weight; its near side, warmer and stiffer, pulls harder than its far side, so it curls a little toward the lamp and tilts the weight. Cooling after the lamp goes off, it lets the weight down again; the loop starts over as the lamp comes back on, the band then not quite back at the room's temperature.

## Pulling the band

Run the other way, the same energy warms the band when it is pulled. With the lamp off, a hand jerks the weight down as hard as its own weight again: the band stretches at nearly fixed entropy, and its triangles warm by their equation of state, most at its top, where it stretches most. Held, it gives that warmth to the room. Let go, it shortens, and its triangles cool below the room's temperature before it warms back. This cell takes about ten seconds, most of it stepping the band.

In [ ]:
# The hand's pull: a quick ramp up, held while the warmth leaks to the room, a quick ramp down, and the band left.
ramp = (1 - np.cos(np.pi * np.arange(scenarios.n_ramp) / scenarios.n_ramp)) / 2
pulling = np.concatenate([ramp, np.ones(scenarios.n_hold), ramp[::-1], np.zeros(scenarios.n_hold)])
pulled_states = show_states(run(settled, np.zeros_like(pulling), pulling, n_stride), render.animate_pulled,
                            topology, in_weight, room_temperature, scenarios.pull_swing, scenarios.pull_drop,
                            name="rubber_band_pulled", duration_ms=scenarios.frame_duration_ms)

The colours are the same palette as under the lamp, grey at the room's temperature, towards red where the band is warmer and towards blue where it is cooler, but a pull warms the band about a thousandth as much as the lamp does, so here the palette spans a thousandth of the lamp's range. The band is red through the pull, fading as the room takes the warmth, blue once it is let go, and fading back.

In [ ]:
# checks
# At the last state's stretch, the band's pull on the support is proportional to its temperature: twice the
# temperatures, the points carried along the entropy by the heat capacity times the logarithm of two at every cell,
# twice the pull.
last_points, *_ = states[-1]
support = topology.reading(held)
warmer_points = last_points + heat_capacity * cells * np.log(2) * entropy     # Point-[vertices]
held_potentials = [support * potentials(points) for points in (last_points, warmer_points)]   # Plane+[held]
support_pulls = [((plane & entropy) * mv.s - plane).sites.sum() & up for plane in held_potentials]
assert np.isclose((support_pulls[1] / support_pulls[0]).to_array(), 2.0, rtol=1e-12)
# The weight rises at every frame while the lamp is on, and once it is off comes back down more than three quarters of
# the way to where it hung.
weight_vertices = topology.reading(np.unique(topology.elements[2][in_weight]))
heights = stack([(mv.y & (weight_vertices * points)).sites.mean() for points, *_ in states]).to_array()   # [frames]
lamp_on = np.array([on for *_, on in states]) > 0                              # [frames]
lifted = heights[lamp_on]                                                     # [lit frames]
assert np.all(np.diff(lifted) > 0)
assert lifted[-1] - heights[-1] > 3 / 4 * (lifted[-1] - heights[0])
# Pulled, the band warms above the room's temperature, and let go, it cools below it.
pulled_changes = np.array([(temperatures - room_temperature).sites.mean().to_array()
                           for _, temperatures, _ in pulled_states])
let_go = len(pulled_changes) // 2
assert pulled_changes[:let_go].max() > 0 > pulled_changes[let_go:].min()
# Both bands end near the room's temperature, within a fifth of their largest change from it.
lamp_changes = np.array([(temperatures - room_temperature).sites.mean().to_array() for _, temperatures, _ in states])
for changes in (lamp_changes, pulled_changes):
    assert abs(changes[-1]) < np.abs(changes).max() / 5